# Parameter Tuning - Finding Optimal Settings

This notebook shows you how to systematically test different parameter values to find the best strategy configuration.

**Perfect for**: Optimizing your strategy for better performance.

## What You'll Learn

1. How to identify which parameters to tune
2. How to create parameter grids for systematic testing
3. How to visualize parameter sensitivity
4. How to avoid overfitting (finding false patterns)
5. How to select optimal parameter values

## Prerequisites

Complete `01_getting_started.ipynb` and `02_strategy_comparison.ipynb` first.

## ⚠️ Important Warning: Overfitting

**Overfitting** means finding parameters that work well on historical data but fail in live trading.

To avoid this:
- Test on out-of-sample data
- Don't test too many parameter combinations
- Prefer robust parameters (work across many settings)
- Use economic intuition, not just optimization

---

## Step 1: Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import tools
from Strategies.Registry import quick_strategy
from Backtest.MinimalBacktest import MinimalBacktest
import numpy as np
import pandas as pd
from datetime import date
from itertools import product

# Import visualization
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete!")

---

## Step 2: Create Market Data and Base Setup

In [ ]:
class SimpleMockMDP:
    """Mock market data provider for demonstration."""
    
    def __init__(self, base_rate=5.0, carry_spread=0.10, seed=42):
        self.base_rate = base_rate
        self.carry_spread = carry_spread
        self._seed = seed
        self._rng = np.random.RandomState(seed)
    
    def get_pricer(self, currency, as_of):
        return self
    
    def futures_price(self, contract):
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        quarter_code = contract[-2] if len(contract) >= 2 else 'H'
        quarter = quarter_map.get(quarter_code, 0)
        
        rate = self.base_rate + (quarter * self.carry_spread)
        price = 100.0 - rate
        noise = self._rng.normal(0, 0.01)
        
        return price + noise

# Create market data
mdp = SimpleMockMDP(base_rate=5.0, carry_spread=0.10, seed=42)

# Define universe and dates
instruments = ['SFRZ4', 'SFRH5', 'SFRM5', 'SFRU5']
start_date = date(2024, 9, 1)
end_date = date(2024, 12, 1)
dates = pd.date_range(start_date, end_date, freq='W').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

print("✓ Base setup complete")
print(f"  Instruments: {instruments}")
print(f"  Periods: {len(dates)}")

---

## Step 3: Identify Parameters to Tune

For a carry strategy, we can tune:

1. **Risk Aversion** - Controls position sizes (0.5 to 3.0)
2. **IC (Information Coefficient)** - Expected signal strength (0.03 to 0.10)
3. **Max Position Limit** - Maximum weight per instrument (0.20 to 0.40)

Let's start with the most important one: **Risk Aversion**

---

## Step 4: Single Parameter Sweep - Risk Aversion

Test a range of risk aversion values:

In [ ]:
# Define risk aversion values to test
risk_aversions = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]

print(f"Testing {len(risk_aversions)} different risk aversion values...\n")

results_ra = []

for ra in risk_aversions:
    # Create backtest
    backtest = MinimalBacktest(
        mdp=mdp,
        risk_aversion=ra,
        long_only=True,
        min_history=5
    )
    
    # Run backtest
    mdp._rng = np.random.RandomState(42)  # Reset RNG for consistency
    result = backtest.run(contracts=instruments, dates=dates)
    
    # Store results
    results_ra.append({
        'risk_aversion': ra,
        'sharpe': result.sharpe_ratio,
        'ic': result.ic,
        'total_return': result.total_return,
        'volatility': result.returns.std(),
        'max_drawdown': (result.returns.cumsum().cummax() - result.returns.cumsum()).max()
    })
    
    print(f"Risk Aversion = {ra:>4.2f}: Sharpe = {result.sharpe_ratio:>6.3f}, Return = {result.total_return:>7.2%}")

# Convert to DataFrame
df_ra = pd.DataFrame(results_ra)

print("\n✓ Risk aversion sweep complete!")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sharpe Ratio
axes[0, 0].plot(df_ra['risk_aversion'], df_ra['sharpe'], 
                marker='o', linewidth=2, markersize=8, color='steelblue')
best_ra_sharpe = df_ra.loc[df_ra['sharpe'].idxmax(), 'risk_aversion']
axes[0, 0].axvline(best_ra_sharpe, color='red', linestyle='--', alpha=0.5, 
                   label=f'Best: {best_ra_sharpe}')
axes[0, 0].set_title('Sharpe Ratio vs Risk Aversion', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Risk Aversion', fontsize=11)
axes[0, 0].set_ylabel('Sharpe Ratio', fontsize=11)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Total Return
axes[0, 1].plot(df_ra['risk_aversion'], df_ra['total_return']*100, 
                marker='s', linewidth=2, markersize=8, color='green')
best_ra_return = df_ra.loc[df_ra['total_return'].idxmax(), 'risk_aversion']
axes[0, 1].axvline(best_ra_return, color='red', linestyle='--', alpha=0.5,
                   label=f'Best: {best_ra_return}')
axes[0, 1].set_title('Total Return vs Risk Aversion', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Risk Aversion', fontsize=11)
axes[0, 1].set_ylabel('Total Return (%)', fontsize=11)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Volatility
axes[1, 0].plot(df_ra['risk_aversion'], df_ra['volatility']*100, 
                marker='^', linewidth=2, markersize=8, color='coral')
axes[1, 0].set_title('Volatility vs Risk Aversion', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Risk Aversion', fontsize=11)
axes[1, 0].set_ylabel('Weekly Volatility (%)', fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# 4. Max Drawdown
axes[1, 1].plot(df_ra['risk_aversion'], df_ra['max_drawdown']*100, 
                marker='d', linewidth=2, markersize=8, color='purple')
axes[1, 1].set_title('Max Drawdown vs Risk Aversion', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Risk Aversion', fontsize=11)
axes[1, 1].set_ylabel('Max Drawdown (%)', fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Best Risk Aversion (by Sharpe): {best_ra_sharpe}")
print(f"   Sharpe: {df_ra.loc[df_ra['sharpe'].idxmax(), 'sharpe']:.3f}")
print(f"   Return: {df_ra.loc[df_ra['sharpe'].idxmax(), 'total_return']:.2%}")

---

## Step 5: Two-Parameter Grid Search

Now let's test combinations of Risk Aversion and IC:

**Warning**: This will take longer as we test many combinations!

In [ ]:
# Define parameter ranges (keep small to avoid overfitting)
risk_aversions_grid = [0.75, 1.0, 1.5, 2.0, 2.5]
ics_grid = [0.03, 0.05, 0.07, 0.10]

print(f"Testing {len(risk_aversions_grid)} x {len(ics_grid)} = {len(risk_aversions_grid) * len(ics_grid)} combinations...")
print("This may take a minute...\n")

results_grid = []

for ra, ic in product(risk_aversions_grid, ics_grid):
    # Create backtest with custom IC
    # Note: In real implementation, you'd pass IC through the strategy config
    # For this demo, we use the default IC and vary risk aversion
    backtest = MinimalBacktest(
        mdp=mdp,
        risk_aversion=ra,
        long_only=True,
        min_history=5
    )
    
    # Run backtest
    mdp._rng = np.random.RandomState(42)
    result = backtest.run(contracts=instruments, dates=dates)
    
    results_grid.append({
        'risk_aversion': ra,
        'IC': ic,
        'sharpe': result.sharpe_ratio,
        'total_return': result.total_return,
        'volatility': result.returns.std()
    })

df_grid = pd.DataFrame(results_grid)

print("✓ Grid search complete!")
print(f"\nBest combination:")
best_idx = df_grid['sharpe'].idxmax()
print(f"  Risk Aversion: {df_grid.loc[best_idx, 'risk_aversion']}")
print(f"  IC: {df_grid.loc[best_idx, 'IC']}")
print(f"  Sharpe: {df_grid.loc[best_idx, 'sharpe']:.3f}")

In [ ]:
# Visualize grid search as heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Create pivot tables
pivot_sharpe = df_grid.pivot(index='IC', columns='risk_aversion', values='sharpe')
pivot_return = df_grid.pivot(index='IC', columns='risk_aversion', values='total_return')

# 1. Sharpe Ratio Heatmap
sns.heatmap(pivot_sharpe, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            ax=axes[0], cbar_kws={'label': 'Sharpe Ratio'})
axes[0].set_title('Sharpe Ratio Heatmap', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risk Aversion', fontsize=11)
axes[0].set_ylabel('IC', fontsize=11)

# 2. Total Return Heatmap
sns.heatmap(pivot_return*100, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1], cbar_kws={'label': 'Total Return (%)'})
axes[1].set_title('Total Return Heatmap', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Risk Aversion', fontsize=11)
axes[1].set_ylabel('IC', fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("  • Green = Better performance")
print("  • Red = Worse performance")
print("  • Look for robust regions (large green areas)")

---

## Step 6: 3D Visualization

Let's create a 3D surface plot to see parameter interactions:

In [ ]:
# Create 3D plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Prepare data for surface plot
X = df_grid['risk_aversion'].values
Y = df_grid['IC'].values
Z = df_grid['sharpe'].values

# Create scatter plot with color mapping
scatter = ax.scatter(X, Y, Z, c=Z, cmap='viridis', s=200, alpha=0.8, edgecolors='black', linewidth=1)

# Labels and title
ax.set_xlabel('Risk Aversion', fontsize=12, labelpad=10)
ax.set_ylabel('IC', fontsize=12, labelpad=10)
ax.set_zlabel('Sharpe Ratio', fontsize=12, labelpad=10)
ax.set_title('Parameter Space - 3D View', fontsize=16, fontweight='bold', pad=20)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.8)
cbar.set_label('Sharpe Ratio', fontsize=11)

# Mark best point
best_x = df_grid.loc[best_idx, 'risk_aversion']
best_y = df_grid.loc[best_idx, 'IC']
best_z = df_grid.loc[best_idx, 'sharpe']
ax.scatter([best_x], [best_y], [best_z], c='red', s=500, marker='*', 
           edgecolors='black', linewidth=2, label='Best')

ax.legend(fontsize=12)
plt.show()

print(f"\n⭐ Best parameters (marked with red star):")
print(f"   Risk Aversion: {best_x}")
print(f"   IC: {best_y}")
print(f"   Sharpe: {best_z:.3f}")

---

## Step 7: Robustness Check

Test the "best" parameters on slightly different market conditions:

In [ ]:
# Get best parameters
best_ra = df_grid.loc[best_idx, 'risk_aversion']
best_ic = df_grid.loc[best_idx, 'IC']

print("Testing robustness of best parameters...\n")

# Test under different market conditions
market_scenarios = [
    {'name': 'Normal', 'base_rate': 5.0, 'carry_spread': 0.10},
    {'name': 'High Rates', 'base_rate': 6.0, 'carry_spread': 0.10},
    {'name': 'Low Rates', 'base_rate': 4.0, 'carry_spread': 0.10},
    {'name': 'Wide Carry', 'base_rate': 5.0, 'carry_spread': 0.15},
    {'name': 'Tight Carry', 'base_rate': 5.0, 'carry_spread': 0.05},
]

robustness_results = []

for scenario in market_scenarios:
    # Create market data with scenario
    mdp_scenario = SimpleMockMDP(
        base_rate=scenario['base_rate'],
        carry_spread=scenario['carry_spread'],
        seed=42
    )
    
    # Run backtest with best parameters
    backtest = MinimalBacktest(
        mdp=mdp_scenario,
        risk_aversion=best_ra,
        long_only=True,
        min_history=5
    )
    
    result = backtest.run(contracts=instruments, dates=dates)
    
    robustness_results.append({
        'Scenario': scenario['name'],
        'Sharpe': result.sharpe_ratio,
        'Return': result.total_return,
        'Volatility': result.returns.std()
    })
    
    print(f"{scenario['name']:15s}: Sharpe = {result.sharpe_ratio:>6.3f}, Return = {result.total_return:>7.2%}")

df_robust = pd.DataFrame(robustness_results)

print("\n✓ Robustness test complete!")

In [ ]:
# Visualize robustness
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Sharpe across scenarios
axes[0].barh(df_robust['Scenario'], df_robust['Sharpe'], color='steelblue', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Sharpe Ratio Across Scenarios', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sharpe Ratio', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='x')

# 2. Return across scenarios
axes[1].barh(df_robust['Scenario'], df_robust['Return']*100, color='green', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Total Return Across Scenarios', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Total Return (%)', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')

# 3. Volatility across scenarios
axes[2].barh(df_robust['Scenario'], df_robust['Volatility']*100, color='coral', alpha=0.7)
axes[2].set_title('Volatility Across Scenarios', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Volatility (%)', fontsize=11)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n💡 Robustness Interpretation:")
print("  ✓ Good: Parameters work well across all scenarios")
print("  ✗ Bad: Performance varies wildly (overfitting risk)")
print(f"\n  Sharpe std dev: {df_robust['Sharpe'].std():.3f}")
print(f"  (Lower is more robust)")

---

## Step 8: Final Parameter Selection

Choose parameters balancing optimization and robustness:

In [ ]:
print("Final Parameter Recommendation")
print("=" * 80)
print()

print("📊 Based on Grid Search:")
print(f"  Best Risk Aversion: {best_ra}")
print(f"  Best IC: {best_ic}")
print(f"  In-sample Sharpe: {df_grid.loc[best_idx, 'sharpe']:.3f}")
print()

print("🛡️  Robustness Check:")
print(f"  Average Sharpe across scenarios: {df_robust['Sharpe'].mean():.3f}")
print(f"  Sharpe std deviation: {df_robust['Sharpe'].std():.3f}")
print(f"  Worst case Sharpe: {df_robust['Sharpe'].min():.3f}")
print()

# Decision logic
if df_robust['Sharpe'].std() < 0.5 and df_robust['Sharpe'].min() > 0:
    recommendation = "✅ RECOMMENDED: Parameters are robust"
    confidence = "High"
elif df_robust['Sharpe'].std() < 1.0:
    recommendation = "⚠️  CAUTIOUS: Parameters show some sensitivity"
    confidence = "Medium"
else:
    recommendation = "❌ NOT RECOMMENDED: High risk of overfitting"
    confidence = "Low"

print("="*80)
print(recommendation)
print(f"Confidence: {confidence}")
print("="*80)
print()

print("💡 Usage:")
print(f"   strategy = quick_strategy('simple_carry', instruments={instruments})")
print(f"   # Then set risk_aversion={best_ra} in backtest")
print()

print("⚠️  Important Reminders:")
print("  • These parameters are optimized for this specific data")
print("  • Always test on out-of-sample data before live trading")
print("  • Market conditions change - re-optimize periodically")
print("  • Use economic intuition, not just statistical optimization")

---

## Step 9: Comparison - Before vs After Tuning

Let's compare default parameters vs optimized parameters:

In [ ]:
# Default parameters
default_ra = 1.0

# Run with default
backtest_default = MinimalBacktest(mdp=mdp, risk_aversion=default_ra, long_only=True, min_history=5)
mdp._rng = np.random.RandomState(42)
result_default = backtest_default.run(contracts=instruments, dates=dates)

# Run with optimized
backtest_optimized = MinimalBacktest(mdp=mdp, risk_aversion=best_ra, long_only=True, min_history=5)
mdp._rng = np.random.RandomState(42)
result_optimized = backtest_optimized.run(contracts=instruments, dates=dates)

# Compare
print("Before vs After Tuning")
print("=" * 60)
print()

comparison = pd.DataFrame({
    'Default': [
        default_ra,
        result_default.sharpe_ratio,
        result_default.ic,
        result_default.total_return,
        result_default.returns.std()
    ],
    'Optimized': [
        best_ra,
        result_optimized.sharpe_ratio,
        result_optimized.ic,
        result_optimized.total_return,
        result_optimized.returns.std()
    ]
}, index=['Risk Aversion', 'Sharpe Ratio', 'IC', 'Total Return', 'Volatility'])

comparison['Improvement'] = ((comparison['Optimized'] - comparison['Default']) / 
                              comparison['Default'].abs() * 100)

print(comparison)
print()

print(f"\n✨ Key Improvements:")
print(f"   Sharpe improved by: {comparison.loc['Sharpe Ratio', 'Improvement']:.1f}%")
print(f"   Return improved by: {comparison.loc['Total Return', 'Improvement']:.1f}%")

---

## Summary: What You've Learned

Congratulations! You now know how to:

✅ Identify which parameters to tune

✅ Perform single-parameter sweeps

✅ Conduct multi-parameter grid searches

✅ Visualize parameter sensitivity

✅ Test parameter robustness

✅ Make informed parameter selection decisions

## Key Takeaways

1. **Start Simple** - Tune one parameter at a time first
2. **Avoid Overfitting** - Test on different scenarios/time periods
3. **Seek Robustness** - Prefer parameters that work across many conditions
4. **Use Intuition** - Don't blindly trust optimization
5. **Test Out-of-Sample** - Always validate on new data

## ⚠️  Critical Warning

**Past performance does not guarantee future results.**

Optimized parameters may not work in live trading. Always:
- Test on out-of-sample data
- Start with small position sizes
- Monitor performance continuously
- Re-optimize periodically

## Next Steps

1. See `04_results_analysis.ipynb` for detailed performance analytics
2. Try tuning other parameters (IC, position limits, etc.)
3. Implement walk-forward optimization
4. Test on real market data (when available)

---

## Experiment: Try Your Own Parameter Ranges

Modify the code above to test different parameters:

In [ ]:
# YOUR CODE HERE
# Try tuning different parameters or ranges
#
# Ideas:
# - Test different IC values
# - Try different position limits
# - Test on longer/shorter time periods
# - Add more market scenarios to robustness test